# 02 — Feature Extraction

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import lower, trim

In [ ]:
spark = SparkSession.builder.appName("hit-predictor").getOrCreate()

In [ ]:
billboard = pd.read_csv("../data/raw/hot100.csv")
billboard = billboard.rename(columns={
    "Song": "title",
    "Artist": "artist",
    "Peak Position": "peak_pos",
    "Weeks in Charts": "weeks_on_chart"
})
billboard["label"] = (billboard["peak_pos"] <= 10).astype(int)
billboard = billboard[["title", "artist", "label"]].drop_duplicates(subset=["title", "artist"])

spotify = pd.read_csv("../data/raw/dataset.csv")
spotify = spotify.rename(columns={"track_name": "title", "artists": "artist"})
spotify = spotify.drop_duplicates(subset=["title", "artist"])

In [ ]:
billboard_sdf = spark.createDataFrame(billboard)
spotify_sdf = spark.createDataFrame(spotify)

for sdf, name in [(billboard_sdf, "billboard"), (spotify_sdf, "spotify")]:
    sdf.withColumn("title", lower(trim(sdf["title"]))) \
       .withColumn("artist", lower(trim(sdf["artist"]))) \
       .createOrReplaceTempView(name)

In [ ]:
FEATURE_COLS = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

cols = ", ".join(f"s.{c}" for c in FEATURE_COLS)
merged_sdf = spark.sql(f"""
    SELECT b.title, b.artist, b.label, {cols}
    FROM billboard b
    INNER JOIN spotify s ON b.title = s.title AND b.artist = s.artist
""")
merged_sdf = merged_sdf.dropna()
print((merged_sdf.count(), len(merged_sdf.columns)))

In [ ]:
merged_sdf.toPandas().to_csv("../data/processed/features.csv", index=False)